In [2]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [3]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [5]:
agmr = 'AGMR'
market = 'TO'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=agmr,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [6]:
agmr_timeseries_df = request_api.get_stock_time_series_data(
    code=agmr,
    market=market,
    start=start,
    end=end
)
agmr_timeseries_df

取得件数: 1066


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1163051,AGMR,TO,2022-02-02,0.455,0.470,0.400,0.460,705300,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1163052,AGMR,TO,2022-02-03,0.440,0.470,0.440,0.460,224100,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1163053,AGMR,TO,2022-02-04,0.440,0.440,0.425,0.430,352800,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1163054,AGMR,TO,2022-02-07,0.445,0.470,0.440,0.470,869800,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1163055,AGMR,TO,2022-02-08,0.450,0.455,0.440,0.445,257500,0.453,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,1164112,AGMR,TO,2026-04-27,3.740,3.880,3.650,3.760,210900,3.916,...,4.204829,3.640771,False,NaN,3.9228,NaN,-0.031594,NaN,NaN,False
1062,1164113,AGMR,TO,2026-04-28,3.600,3.690,3.580,3.670,90300,3.792,...,4.197914,3.673286,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1063,1164114,AGMR,TO,2026-04-29,3.400,3.640,3.350,3.640,182200,3.730,...,4.197956,3.672444,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1064,1164115,AGMR,TO,2026-04-30,3.530,3.550,3.380,3.450,258500,3.658,...,4.173333,3.632267,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [8]:
def stock_prices_and_gold_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_gold: pd.DataFrame | None = None,
        df_silver: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # 金価格を統合
    if df_gold is not None:
        df_gold_tmp = df_gold.copy() if df_gold is not None else pd.DataFrame()
        if "date" not in df_gold_tmp.columns:
            df_gold_tmp = df_gold_tmp.reset_index()
        df_gold_tmp["date"] = pd.to_datetime(df_gold_tmp["date"])
        df_gold_tmp = df_gold_tmp.set_index("date")
        df_gold_tmp = df_gold_tmp.loc[start:end]

    # 銀価格を統合
    if df_silver is not None:
        df_silver_tmp = df_silver.copy() if df_silver is not None else pd.DataFrame()
        if "date" not in df_silver_tmp.columns:
            df_silver_tmp = df_silver_tmp.reset_index()
        df_silver_tmp["date"] = pd.to_datetime(df_silver_tmp["date"])
        df_silver_tmp = df_silver_tmp.set_index("date")
        df_silver_tmp = df_silver_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_gold is not None:
        df["MA5_GOLD"] = df_gold_tmp["ma5"].reindex(df.index)
        df["MA25_GOLD"] = df_gold_tmp["ma25"].reindex(df.index)
    if df_silver is not None:
        df["MA5_SILVER"] = df_silver_tmp["ma5"].reindex(df.index)
        df["MA25_SILVER"] = df_silver_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- GOLD（右軸） ---
    if df_gold is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_GOLD"],
                name="GOLD_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_GOLD"],
                name="GOLD_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- SILVER（左軸） ---
    if df_silver is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SILVER"],
                name="SILVER_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SILVER"],
                name="SILVER_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [9]:
name = "Silver Mountain Resources Inc"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_gold_prices(
    code=agmr,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_gold=None,
    df_silver=None
)
fig.show()

取得件数: 1062


In [10]:
response = request_api.update_corp_finance_data(
    code=agmr,
    market=market
)
response

POST response: {'detail': 'An unexpected error occurred'}


{}

In [12]:
agmr_financials_data = request_api.get_corp_financials_data(code=agmr, market=market)
agmr_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=agmr, market=market)
agmr_cash_flow_data = request_api.get_corp_cash_flow_data(code=agmr, market=market)
agmr_earnings_data = request_api.get_corp_earnings_data(code=agmr, market=market)
agmr_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=agmr, market=market)

GET response: {'detail': 'Corporate finance data not found'}
GET response: {'detail': 'Balance sheet data not found'}
GET response: {'detail': 'Cash flow data not found'}
GET response: {'detail': 'Earnings data not found'}
GET response: {'detail': 'Quarterly earnings data not found'}


In [13]:
# ４年分の財務データ
agmr_financials_data_df = pd.DataFrame(agmr_financials_data['results'])
# ４年分のバランスシート
agmr_balance_sheet_data_df = pd.DataFrame(agmr_balance_sheet_data['results'])
# ４年分のキャッシュフロー
agmr_cash_flow_data_df = pd.DataFrame(agmr_cash_flow_data['results'])
# ４年分の収益データ
agmr_earnings_data_df = pd.DataFrame(agmr_earnings_data['results'])
# ４年分の四半期収益データ
agmr_quarterly_earnings_data_df = pd.DataFrame(agmr_quarterly_earnings_data['results'])

KeyError: 'results'

In [ ]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_market_data.stock_prices_and_market_data(
    code=agmr,
    market=market,
    bs_df=agmr_balance_sheet_data_df
)

In [ ]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = agmr,
    market = market,
)
financial_df

In [ ]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=agmr, market=market)

In [ ]:
#md_file_path =pdf_to_md.pdf_url_to_markdown(
#    pdf_url="https://www.ayagoldsilver.com/_resources/financials/2025/AIF-2025.pdf",
#    directory_path="/workspace/data",
#)
#md_file_path

In [ ]:

#md_file_path = webpage_to_markdown.webpage_to_markdown(
#    url = '',
#    directory_path = '/workspace/data'
#)
#md_file_path

In [14]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://agmr.ca/wp-content/uploads/2025/01/Final_-_7_-_AR_Tech_Report.pdf",
    directory_path="/workspace/data",
)
md_file_path

markitdown failed (rc=-9). Falling back to pdftotext if available. stderr:
onnxruntime cpuid_info warning: Unknown CPU vendor. cpuinfo_vendor value: 0
Generated (pdftotext fallback): /workspace/data/Final_-_7_-_AR_Tech_Report.pdf.md


'/workspace/data/Final_-_7_-_AR_Tech_Report.pdf.md'

In [19]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://agmr.ca/wp-content/uploads/2023/09/Technical_Report_El-Milagro.pdf",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/Technical_Report_El-Milagro.pdf.md


'/workspace/data/Technical_Report_El-Milagro.pdf.md'

In [20]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://www.sedarplus.ca/csa-party/viewInstance/resource.html?node=W5082&drmKey=0a6d8dc389c16fbe&drr=ss9fc506fd9bcb9fad54dbe4b3b844ea36b6fa0e753e2395fc63b42478cc00cee067a6d9aae480b9f903b3737da55c7a08ux&id=0c11f8b7998bcd9614485dcad0f7e7f9cec6f716bfac7c0a",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/resource.html.md


'/workspace/data/resource.html.md'

In [18]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://www.sedarplus.ca/csa-party/viewInstance/resource.html?node=W5153&drmKey=9d93126107f803ce&drr=ss9fc506fd9bcb9fad54dbe4b3b844ea36b6fa0e753e2395fc63b42478cc00cee067a6d9aae480b9f903b3737da55c7a08ux&id=0c11f8b7998bcd9614485dcad0f7e7f9cec6f716bfac7c0a",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/resource.html.md


'/workspace/data/resource.html.md'

In [17]:
md_file_path =pdf_to_md.pdf_url_to_markdown(
    pdf_url="https://www.sedarplus.ca/csa-party/viewInstance/resource.html?node=W5224&drmKey=80ca9a2f2cc5f931&drr=ss9fc506fd9bcb9fad54dbe4b3b844ea36b6fa0e753e2395fc63b42478cc00cee067a6d9aae480b9f903b3737da55c7a08ux&id=0c11f8b7998bcd9614485dcad0f7e7f9cec6f716bfac7c0a",
    directory_path="/workspace/data",
)
md_file_path

Generated: /workspace/data/resource.html.md


'/workspace/data/resource.html.md'